In [ ]:
from pathlib import Path
import torch
import torchaudio
from torch.utils.data import Dataset

COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
LABEL2IDX = {label: i fo

class SpeechCommandsSubset(Dataset):
    def __init__(self, root="data", subset="training", sample_rate=16000):
        self.ds = torchaudio.datasets.SPEECHCOMMANDS(
            root=root,
            url="speech_commands_v0.02",
            download=True,
            subset=subset,
        )
        self.sample_rate = sample_rate
        self.items = []

        for i in range(len(self.ds)):
            waveform, sr, label, speaker_id, utt = self.ds[i]
            if label in LABEL2IDX:
                self.items.append((waveform, sr, label))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        waveform, sr, label = self.items[idx]

        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(waveform, sr, self.sample_rate)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        target_len = self.sample_rate
        if waveform.shape[1] < target_len:
            pad = target_len - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :target_len]

        return waveform, LABEL2IDX[label]

In [ ]:
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.net(x)
        x = x.flatten(1)
        return self.fc(x)

In [ ]:
import torch
import torchaudio

class SpectrogramFrontend(torch.nn.Module):
    def __init__(self, n_fft=400, hop_length=160, n_mels=64, sample_rate=16000):
        super().__init__()
        self.spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
        )

    def forward(self, waveform):
        x = self.spec(waveform)
        x = torch.log(x + 1e-6)
        return x

In [ ]:
import argparse
import torch
from torch.utils.data import DataLoader
from dataset import SpeechCommandsSubset
from features import SpectrogramFrontend
from model import SmallCNN

def collate_fn(batch):
    xs, ys = zip(*batch)
    x = torch.stack(xs, dim=0)
    y = torch.tensor(ys, dtype=torch.long)
    return x, y

def evaluate(model, frontend, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            feats = frontend(waveforms).unsqueeze(1)
            logits = model(feats)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    return correct / total

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", type=str, default="data")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    train_ds = SpeechCommandsSubset(root=args.data_root, subset="training")
    val_ds = SpeechCommandsSubset(root=args.data_root, subset="validation")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    device = torch.device(args.device)
    frontend = SpectrogramFrontend().to(device)
    model = SmallCNN(num_classes=10).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(args.epochs):
        model.train()
        for waveforms, labels in train_loader:
            waveforms = waveforms.to(device)
            labels = labels.to(device)

            feats = frontend(waveforms).unsqueeze(1)
            logits = model(feats)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_acc = evaluate(model, frontend, val_loader, device)
        print(f"epoch={epoch+1} val_acc={val_acc:.4f}")

    torch.save(model.state_dict(), "speech_command_cnn.pt")

if __name__ == "__main__":
    main()